# 02 — Modeling, Evaluation & Error Analysis

This notebook loads artifacts from `scripts/train_eval.py` and walks through:
- model comparison (MAE / RMSE / NASA score)
- error analysis (residuals, early vs late predictions, worst engines)
- model interpretation (feature importance)


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

reports = ROOT / 'reports'
figures = reports / 'figures'
metrics = json.loads((reports / 'metrics.json').read_text())
metrics.keys()


## Validation and test comparison


In [ ]:
val = pd.DataFrame(metrics['validation'])
test = pd.DataFrame(metrics['test'])
print('Validation metrics')
display(val)
print('Official test metrics (last cycle per engine)')
display(test)
display(Image(filename=str(figures / 'model_comparison.png')))


## Predicted vs actual / residuals / error vs RUL (best model)


In [ ]:
best = metrics['error_analysis']['best_model_by_rmse']
slug = best.lower()
print('Best model by test RMSE:', best)
for name in [f'{slug}_pred_vs_actual.png', f'{slug}_residuals.png', f'{slug}_error_vs_rul.png']:
    display(Image(filename=str(figures / name)))


## Per-engine error analysis
Late predictions (`pred > true`) are costlier in maintenance settings; the NASA asymmetric score penalizes them more heavily.


In [ ]:
err = pd.read_csv(reports / 'per_engine_errors.csv')
display(err.head(10))
ea = metrics['error_analysis']
print('Mean signed error (pred-true):', ea['mean_signed_error'])
print('% late predictions:', round(ea['pct_late_predictions'], 1))
print('% early predictions:', round(ea['pct_early_predictions'], 1))
print('Worst 5 engines:', ea['worst_5_engines'])


## Feature importance


In [ ]:
display(Image(filename=str(figures / 'rf_importance.png')))
display(Image(filename=str(figures / 'xgb_importance.png')))
pd.read_csv(reports / 'rf_feature_importance.csv').head(10)


## Written findings
- Tree models (RF / XGBoost) should beat linear regression when sensor interactions and non-linear degradation matter.
- Errors often grow for engines with unusual degradation speed or very high true RUL (early truncation).
- Rolling statistics of high-signal sensors (e.g. sensors around HPC health) typically dominate importance rankings.
- Always interpret NASA score together with MAE/RMSE: a model can look decent on MAE but worse on late-prediction risk.
